In [5]:
!pip install -q scikit-learn matplotlib

In [6]:
!pip install -q tensorflow

In [7]:
# import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
# import mediapipe as mp

## preprocess
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

## model
# import tensorflow as tf
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import LSTM, Dense, Input, Bidirectional, Dropout, LayerNormalization, TimeDistributed, Attention, Concatenate
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam


## train eval
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [9]:
import tensorflow as tf

In [ ]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic
mp_face_mesh = mp.solutions.face_mesh

In [ ]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [ ]:
STYLES = {
    "face": ((80, 110, 10), (80, 256, 121), 1, 1), 
    # "pose": ((80, 22, 10), (80, 44, 121), 2, 4),
    "left_hand": ((121, 22, 76), (121, 44, 250), 2, 4),
    "right_hand": ((245, 117, 66), (245, 66, 230), 2, 4)
}

def draw_styled_landmarks(image: any, results: any) -> None:
    landmarks_mapping = {
        "face": (results.face_landmarks, mp_face_mesh.FACEMESH_CONTOURS),
        # "pose": (results.pose_landmarks, mp_holistic.POSE_CONNECTIONS),
        "left_hand": (results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS),
        "right_hand": (results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS),
    }

    for key, (landmarks, connections) in landmarks_mapping.items():
        if landmarks:
            primary_color, secondary_color, thickness, radius = STYLES[key]
            mp_drawing.draw_landmarks(
                image, 
                landmarks, 
                connections,
                mp_drawing.DrawingSpec(color=primary_color, thickness=thickness, circle_radius=radius),
                mp_drawing.DrawingSpec(color=secondary_color, thickness=thickness, circle_radius=radius // 2)
            )

In [ ]:
# cap = cv2.VideoCapture(0)
# # Set mediapipe model 
# with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
#     while cap.isOpened():

#         ret, frame = cap.read()

#         frame = cv2.flip(frame, 1)

#         image, results = mediapipe_detection(frame, holistic)
        
#         draw_styled_landmarks(image, results)

#         cv2.imshow('OpenCV Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q') or cv2.getWindowProperty('OpenCV Feed', cv2.WND_PROP_VISIBLE) < 1:
#             break
#     cap.release()
#     cv2.destroyAllWindows()
#     cv2.waitKey(1)

## Extract Features

In [ ]:
def extract_landmarks(results):
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([face, lh, rh])

## Set Up Image Datasets Directory

In [10]:
DATA_PATH = os.path.join('/kaggle/input/fp-kcvanguard-dataset3/dataset') 

# label = open('label.txt', 'r').readline().split()
# label2 = open('label2.txt', 'r').readline().split()

# no_sequences = 30

sequence_length = 19

start_folder = 30

In [ ]:
# for item in label3:
#     try:
#         os.makedirs(os.path.join(DATA_PATH, item))
#         for num_folder in range(start_folder):
#             try:
#                 os.makedirs(os.path.join(DATA_PATH, item, item+str(num_folder+1)))
#             except:
#                 pass
#     except:
#         pass        

## Extract Datasets

In [ ]:
label3 = ['siapa']

In [ ]:
# cap = cv2.VideoCapture(1)

# begin_extract = False

# def mouse_callback(event, x, y, flags, param):
#     global begin_extract
#     if event == cv2.EVENT_LBUTTONDOWN:
#         if 50 <= x <= 250 and 50 <= y <= 100:  
#             begin_extract = True 

# cv2.namedWindow('OpenCV Feed')
# cv2.setMouseCallback('OpenCV Feed', mouse_callback)

# flag = 1

# with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
#     for item in label3:
#         for num_folder in range(start_folder):
#             for frame_num in range(sequence_length):

#                 ret, frame = cap.read()

#                 frame = cv2.flip(frame, 1)

#                 image, results = mediapipe_detection(frame, holistic)

#                 draw_styled_landmarks(image, results)

#                 while not begin_extract:
#                     ret, frame = cap.read()

#                     frame = cv2.flip(frame, 1)

#                     image, results = mediapipe_detection(frame, holistic)
                    
#                     draw_styled_landmarks(image, results)
                    
#                     cv2.rectangle(image, (50, 50), (250, 100), (0, 255, 0), -1)
#                     cv2.putText(image, "Start Collection", (60, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                
#                     cv2.imshow('OpenCV Feed', image)
#                     if cv2.waitKey(10) & 0xFF == ord('q') or cv2.getWindowProperty('OpenCV Feed', cv2.WND_PROP_VISIBLE) < 1:
#                         flag=0
#                         break
                
#                 if frame_num == 0: 
#                     cv2.waitKey(500)
#                     ret, frame = cap.read()

#                     frame = cv2.flip(frame, 1)
                    
#                     image, results = mediapipe_detection(frame, holistic)
                    
#                     cv2.putText(image, 'STARTING COLLECTION', (120,200), 
#                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
#                     cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(item, num_folder), (15,12), 
#                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
#                     cv2.imshow('OpenCV Feed', image)
#                     cv2.waitKey(500)
#                 else: 
#                     cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(item, num_folder), (15,12), 
#                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
#                     cv2.imshow('OpenCV Feed', image)
                
#                 keypoints = extract_landmarks(results)
#                 npy_path = os.path.join(DATA_PATH, item, item+str(num_folder+1), str(frame_num))
#                 np.save(npy_path, keypoints)

#                 # global begin_extract
#                 # begin_extract = False

#                 if not flag:
#                     break

#                 if cv2.waitKey(10) & 0xFF == ord('q') or cv2.getWindowProperty('OpenCV Feed', cv2.WND_PROP_VISIBLE) < 1:   
#                     flag=0
#                     break
                    
#             begin_extract = False
#             if not flag:
#                 break
#         if not flag:
#             break
        
#     cap.release()
#     cv2.destroyAllWindows()
#     cv2.waitKey(10)
    

## Build Model

In [11]:
label_train = ['hai', 'nama', 'kamu', 'pagi', 'siang', 'malam', 'siapa', 'sudah', 'belum', 'makan','suka', 'selamat', 'aku']

In [12]:
def build_bilstm_model(num_vocabs, num_frames=19, num_landmarks=42):

    input_layer = Input(shape=(num_frames, num_landmarks*3))  
    
    # Pre-processing layers
    x = TimeDistributed(Dense(256, activation='relu', kernel_regularizer=l2(0.001)))(input_layer)
    x = LayerNormalization()(x)
    x = Dropout(0.3)(x)
    
    # First BiLSTM layer
    x = Bidirectional(
        LSTM(256, return_sequences=True, kernel_regularizer=l2(0.001)),
        merge_mode='concat'
    )(x)
    x = LayerNormalization()(x)
    x = Dropout(0.4)(x)
    
    # Second BiLSTM layer with attention
    lstm_out = Bidirectional(
        LSTM(256, return_sequences=True, kernel_regularizer=l2(0.001)),
        merge_mode='concat'
    )(x)
    lstm_out = LayerNormalization()(lstm_out)
    
    # Attention mechanism
    attention = Attention()([lstm_out, lstm_out])
    x = Concatenate()([lstm_out, attention])
    x = Dropout(0.5)(x)
    
    # Third BiLSTM layer
    x = Bidirectional(
        LSTM(256, return_sequences=False, kernel_regularizer=l2(0.001)),
        merge_mode='concat'
    )(x)
    x = LayerNormalization()(x)
    
    # Dense layers
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    
    # Output layer
    output_layer = Dense(num_vocabs, activation='softmax')(x)
    
    # Create model
    model = Model(inputs=input_layer, outputs=output_layer)
    
    # Compile model
    optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Example usage
# model = build_bilstm_model(num_vocabs=50)  # replace 50 with your actual number of vocabulary classes

In [31]:
def train_model(X_train, y_train, X_val, y_val, num_vocabs):

    # Build model
    model = build_bilstm_model(num_vocabs)

    class StopAt100Acc(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            if logs.get('val_accuracy') == 1.0 or logs.get('accuracy') == 1.0:
                print("\nReached 100% val accuracy. Stopping training.")
                self.model.stop_training = True
    
    # Callbacks
    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=15, min_lr=1e-5),
        tf.keras.callbacks.ModelCheckpoint(
            'best_model_handlandmarks.keras',
            save_best_only=True,
            monitor='val_accuracy',
            mode='max'
        ),
        StopAt100Acc()
    ]
    
    # Train model
    with tf.device('/device:GPU:0'):
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=2000,
            batch_size=128,
            callbacks=callbacks,
            verbose=1
        )
    
    return model, history

In [14]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [ ]:
# model.fit(X_train, y_train, epochs=2000, callbacks=[tb_callback])

In [ ]:
# model.summary()

## Dataset Preprocess

In [15]:
label_map = {label:num for num, label in enumerate(label_train)}

In [16]:
label_map

{'hai': 0,
 'nama': 1,
 'kamu': 2,
 'pagi': 3,
 'siang': 4,
 'malam': 5,
 'siapa': 6,
 'sudah': 7,
 'belum': 8,
 'makan': 9,
 'suka': 10,
 'selamat': 11,
 'aku': 12}

In [17]:
sequences, labels = [], []
for action in label_train:
    for sequence in range(start_folder, start_folder*2):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, action+str(sequence+1), "{}.npy".format(frame_num+1)))
            window.append(res[1404:])
        sequences.append(window)
        labels.append(label_map[action])

In [18]:
np.array(sequences).shape

(390, 19, 126)

In [19]:
np.array(labels).shape

(390,)

## Train Model

In [20]:
X = np.array(sequences)
y = to_categorical(labels).astype(int)

In [21]:
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=42, test_size=0.2)

In [22]:
X_train.shape

(312, 19, 126)

In [23]:
y_train.shape

(312, 13)

In [24]:
X_val.shape

(78, 19, 126)

In [25]:
y_val.shape

(78, 13)

In [ ]:
# log_dir = os.path.join('Logs')
# tb_callback = TensorBoard(log_dir=log_dir)

In [30]:
model, result = train_model(X_train = X_train, y_train = y_train, X_val = X_val, y_val = y_val, num_vocabs = len(label_train))

Epoch 1/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 688ms/step - accuracy: 0.0929 - loss: 8.1397 - val_accuracy: 0.2179 - val_loss: 7.6668 - learning_rate: 0.0010
Epoch 2/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.1741 - loss: 7.7699 - val_accuracy: 0.2051 - val_loss: 7.3764 - learning_rate: 0.0010
Epoch 3/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.1443 - loss: 7.5636 - val_accuracy: 0.1923 - val_loss: 7.1765 - learning_rate: 0.0010
Epoch 4/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 236ms/step - accuracy: 0.2313 - loss: 7.3224 - val_accuracy: 0.2821 - val_loss: 6.7567 - learning_rate: 0.0010
Epoch 5/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 239ms/step - accuracy: 0.2727 - loss: 7.0003 - val_accuracy: 0.3333 - val_loss: 6.5922 - learning_rate: 0.0010
Epoch 6/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 229ms/step - accuracy: 0.2323 - loss: 6.8377 - val_accuracy: 0.3590 - val_loss: 6.3886 - learning_rate: 0.0010
Epoch 7/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.2638 - loss: 6.6817 - va

KeyboardInterrupt: 

## Testing

In [ ]:
# model.load_weights("model1.keras")

In [ ]:
# y_preds = model.predict(X_val)
# y_preds = np.argmax(y_preds, axis=1).tolist()
# y_true = np.argmax(y_val, axis=1).tolist()

In [ ]:
# accuracy_score(y_true, y_preds)

## Demo

In [ ]:
colors = [(245,117,16), (117,245,16), (16,117,245)]
def prob_viz(res, labels, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), colors[num], -1)
        cv2.putText(output_frame, labels[num], (0, 85+num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)
        
    return output_frame

In [ ]:
# plt.figure(figsize=(18,18))
# plt.imshow(prob_viz(res, actions, image, colors))

In [ ]:
def extract_landmarks(results):
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([face, lh, rh])

In [ ]:
# 1. New detection variables
sequence = []
sentence = []
predictions = []
threshold = 0.5

cap = cv2.VideoCapture(1)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        frame = cv2.flip(frame, 1)

        image, results = mediapipe_detection(frame, holistic)
        
        draw_styled_landmarks(image, results)
        
        keypoints = extract_landmarks(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        for seq in sequence:
            # print(seq)
            if np.all(seq[1404:] == 0):
                print("acumalak")
                sequence = []
        
        if len(sequence) == sequence_length:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(res)
            print(label3[np.argmax(res)])
            sequence = []
            predictions.append(np.argmax(res))
            
            
        #3. Viz logic
            # if np.unique(predictions[-10:])[0]==np.argmax(res): 
            if res[np.argmax(res)] > threshold: 
                
                if len(sentence) > 0: 
                    if label3[np.argmax(res)] != sentence[-1]:
                        sentence.append(label3[np.argmax(res)])
                else:
                    sentence.append(label3[np.argmax(res)])

            if len(sentence) > 5: 
                sentence = sentence[-5:]

            # Viz probabilities
            # image = prob_viz(res, label3, image, colors)
            
        cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3,30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        cv2.waitKey(10)

        # Break gracefully
        if cv2.getWindowProperty('OpenCV Feed', cv2.WND_PROP_VISIBLE) < 1:
            break
    cap.release()
    cv2.destroyAllWindows()